# PSAT — Performance Benchmark: Numba JIT vs Pure Python

This notebook quantifies the performance gain from using `@numba.njit` to compile
the Euler-Maruyama physics loop to native machine code via LLVM.

**Method:** We implement the same core computation in two ways:
1. **Pure Python** — a plain `for` loop in CPython (no compiled extensions).
2. **Numba JIT** — the `jitted_physics_core` function pre-compiled by Numba.

Both are timed with `timeit` at two particle counts: N = 1 000 and N = 5 000.

In [ ]:
import time
import timeit

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

from psat.constants import g
from psat.engine import jitted_physics_core

## 1 — Shared Setup

Both implementations receive identical inputs so the comparison is apples-to-apples.

In [ ]:
def make_inputs(n: int):
    """Generate random but physically valid particle inputs."""
    rng = np.random.default_rng(42)
    dt = 0.001
    x = rng.uniform(0.0, 0.05, n)
    y = rng.uniform(-0.008, 0.008, n)
    z = rng.uniform(-0.008, 0.008, n)
    ux = 0.5 * (1 - (y**2 + z**2) / 0.01**2)
    uy = np.zeros(n)
    uz = np.zeros(n)
    tau = np.full(n, 3e-5)  # ~5 µm particle
    D = np.full(n, 5e-9)  # brownian diffusivity
    Z = np.zeros(n)  # no charge
    # pre-allocated output buffers
    x_new = np.empty(n)
    y_new = np.empty(n)
    z_new = np.empty(n)
    hit_wall = np.empty(n, dtype=np.bool_)
    hit_bot = np.empty(n, dtype=np.bool_)
    return (
        x,
        y,
        z,
        ux,
        uy,
        uz,
        tau,
        D,
        Z,
        0.0,
        0.0,
        0.0,  # v_th xyz
        0.0,
        0.0,
        0.0,  # E xyz
        dt,
        g,
        0.0,
        0.1,
        0.01,
        0.05,
        np.pi / 6,
        x_new,
        y_new,
        z_new,
        hit_wall,
        hit_bot,
    )

## 2 — Pure Python Baseline

This re-implements the same Euler-Maruyama step as vanilla CPython — no NumPy vectorisation,
no Numba, exactly as a junior developer might write it.

In [ ]:
import math
import random


def python_physics_core(
    x,
    y,
    z,
    ux,
    uy,
    uz,
    tau,
    D,
    Z,
    v_th_x,
    v_th_y,
    v_th_z,
    Ex,
    Ey,
    Ez,
    dt,
    gravity,
    xmin,
    xmax,
    ymax,
    L1,
    theta,
    x_new,
    y_new,
    z_new,
    hit_wall,
    hit_bottom,
):
    """Equivalent pure-Python Euler-Maruyama step (no Numba)."""
    n = len(x)
    R_main_sq = ymax**2
    R_branch_sq = (ymax / math.sqrt(2.0)) ** 2
    cos_t = math.cos(theta)
    tan_t = math.tan(theta)

    for i in range(n):
        v_set_y = -tau[i] * gravity
        tvx = ux[i] + v_th_x + Z[i] * Ex
        tvy = uy[i] + v_th_y + Z[i] * Ey + v_set_y
        tvz = uz[i] + v_th_z + Z[i] * Ez

        sigma = math.sqrt(2.0 * D[i] * dt)
        nx = x[i] + tvx * dt + random.gauss(0, sigma)
        ny = y[i] + tvy * dt + random.gauss(0, sigma)
        nz = z[i] + tvz * dt + random.gauss(0, sigma)

        hw = False
        hb = False
        if nx <= L1:
            if ny * ny + nz * nz >= R_main_sq:
                hw = True
        else:
            xb = nx - L1
            ycu = xb * tan_t
            ycd = -xb * tan_t
            du2 = (ny - ycu) ** 2 * cos_t**2 + nz**2
            dd2 = (ny - ycd) ** 2 * cos_t**2 + nz**2
            if du2 >= R_branch_sq and dd2 >= R_branch_sq:
                hw = True
        if nx >= xmax:
            hb = True

        x_new[i] = nx
        y_new[i] = ny
        z_new[i] = nz
        hit_wall[i] = hw
        hit_bottom[i] = hb

## 3 — Numba JIT Warm-Up

Numba compiles the function on its **first call**.  We run it once with N = 10
to pay the one-time compilation cost, then time the *hot* path.

In [ ]:
# Trigger JIT compilation (excludes warm-up from benchmark)
warmup_args = make_inputs(10)
t_warmup_start = time.perf_counter()
jitted_physics_core(*warmup_args)
t_warmup = time.perf_counter() - t_warmup_start
print(f"Numba JIT compilation (one-time warm-up): {t_warmup:.3f} s")

## 4 — Timed Comparison

In [ ]:
PARTICLE_COUNTS = [1000, 5000]
REPEATS = 5  # number of timing repetitions

results = {}  # {n: {"python": t, "numba": t}}

for n in PARTICLE_COUNTS:
    args = make_inputs(n)

    # ── Pure Python ──────────────────────────────────────────────────────
    t_py = timeit.timeit(lambda: python_physics_core(*args), number=REPEATS) / REPEATS

    # ── Numba JIT (hot path only) ─────────────────────────────────────────
    t_jit = timeit.timeit(lambda: jitted_physics_core(*args), number=REPEATS) / REPEATS

    results[n] = {"python": t_py, "numba": t_jit}
    speedup = t_py / t_jit
    print(
        f"N={n:>5}  |  Python: {t_py * 1000:7.2f} ms  |  "
        f"Numba: {t_jit * 1000:6.3f} ms  |  Speedup: {speedup:.0f}×"
    )

## 5 — Results Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor("#0e1117")

colors = {"python": "#ff6b6b", "numba": "#4f8ef7"}
bar_w = 0.35
ns = PARTICLE_COUNTS
x_pos = np.arange(len(ns))

for ax in axes:
    ax.set_facecolor("#0e1117")
    ax.tick_params(colors="#aaa")
    ax.spines[:].set_color("#333")
    ax.xaxis.label.set_color("#ccc")
    ax.yaxis.label.set_color("#ccc")
    ax.title.set_color("white")

# ── Absolute time (log scale) ─────────────────────────────────────────────────
ax = axes[0]
py_times = [results[n]["python"] * 1000 for n in ns]
jit_times = [results[n]["numba"] * 1000 for n in ns]

b1 = ax.bar(
    x_pos - bar_w / 2, py_times, bar_w, label="Pure Python", color=colors["python"], alpha=0.9
)
b2 = ax.bar(
    x_pos + bar_w / 2, jit_times, bar_w, label="Numba JIT", color=colors["numba"], alpha=0.9
)

ax.set_yscale("log")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"N = {n:,}" for n in ns], color="#ccc")
ax.set_ylabel("Time per step (ms, log scale)")
ax.set_title("Execution Time per Step")
ax.legend(facecolor="#1a1a2e", labelcolor="white")
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.3g"))

# Annotate speedup above each pair
for i, n in enumerate(ns):
    sp = results[n]["python"] / results[n]["numba"]
    ax.text(
        x_pos[i],
        max(py_times[i], jit_times[i]) * 1.4,
        f"{sp:.0f}×",
        ha="center",
        color="#ffd43b",
        fontweight="bold",
        fontsize=11,
    )

# ── Speedup factor ────────────────────────────────────────────────────────────
ax2 = axes[1]
speedups = [results[n]["python"] / results[n]["numba"] for n in ns]
bars = ax2.bar(x_pos, speedups, 0.5, color="#51cf66", alpha=0.9)

for rect, sp in zip(bars, speedups):
    ax2.text(
        rect.get_x() + rect.get_width() / 2,
        rect.get_height() + 1,
        f"{sp:.0f}×",
        ha="center",
        va="bottom",
        color="white",
        fontweight="bold",
    )

ax2.set_xticks(x_pos)
ax2.set_xticklabels([f"N = {n:,}" for n in ns], color="#ccc")
ax2.set_ylabel("Speedup factor (×)")
ax2.set_title("Numba JIT Speedup over Pure Python")
ax2.set_ylim(0, max(speedups) * 1.25)

fig.suptitle(
    "PSAT — Numba JIT vs Pure Python Performance", color="white", fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.savefig("benchmark_results.png", dpi=150, bbox_inches="tight", facecolor="#0e1117")
plt.show()
print("Chart saved as benchmark_results.png")

## 6 — Key Takeaways

| Metric | Pure Python | Numba JIT |
|--------|-------------|------------|
| N=500  | see above   | see above  |
| N=2000 | see above   | see above  |
| Warm-up cost | none | one-time (cached after) |

**Why Numba?**
- The physics core is a tight inner loop over thousands of particles per step.
- CPython's interpreter overhead (bytecode dispatch, refcounting, boxing) dominates at this scale.
- `@numba.njit` compiles the function to LLVM IR → native x86-64 via LLVM, removing all
  interpreter overhead. SIMD auto-vectorisation and `fastmath=True` (permits reordering of
  floating-point ops for throughput) contribute additional gains.
- The compilation cost (~1–3 s) is paid **once** per process start and is then cached to disk
  (`__pycache__/numba_cache/`), making subsequent runs instant.

This demonstrates that choosing the right tooling (not just switching languages) can yield
**orders-of-magnitude** speedups while keeping the code readable Python.